Alerta de ordens pendentes criação de KPI:

Ordens pendentes, quantidades de itens e informações do cliente para o envio de alerta por e-mail e telefone, para aqueles com cadastro completo

In [0]:
%python
bronze_path   = 'bikestore.bronze'
silver_path   = 'bikestore.silver'
gold_path     = 'bikestore.gold'
resource_path = "/Volumes/bikestore/resource/origem"
resource_path_volume = '/Volumes/bikestore/resource/origem'

In [0]:
%python
# criar varias tabelas temporárias de forma prática

silver_map = {
    # View_name --------------- path
    "tmp_silver_customers": f"{silver_path}.customers",
    "tmp_silver_orders": f"{silver_path}.orders",
    # "tmp_silver_products": f"{silver_path}.product"
    }

for view_name, path in silver_map.items():
    (spark.table(path).createOrReplaceTempView(view_name))

In [0]:
select * from tmp_silver_customers limit 4

In [0]:
select * from tmp_silver_orders limit 4

In [0]:
describe tmp_silver_orders

In [0]:
select
  o.customer_id,
  o.order_date,
  sum(o.quantity) as quantity,
  o.store_name,
  source_orders_file,
  source_store_file,
  source_staffs_file,
  source_order_items_file
  -- o.order_status
from
  tmp_silver_orders o
where
  1 = 1
  AND upper(order_status) = "PENDING"
group by 
  o.customer_id,
  o.order_date,
  o.store_name,
  source_orders_file,
  source_store_file,
  source_staffs_file,
  source_order_items_file
  -- o.order_status

In [0]:
select * from tmp_silver_customers

In [0]:
describe tmp_silver_customers

In [0]:
with pending as (
  select
    customer_id,
    order_date,
    sum(quantity) as quantity,
    store_name,
    source_orders_file,
    source_store_file,
    source_staffs_file,
    source_order_items_file
  -- p.order_status
  from
    tmp_silver_orders
  where
    1 = 1
    AND upper(order_status) = "PENDING"
  group by
    customer_id,
    order_date,
    store_name,
    source_orders_file,
    source_store_file,
    source_staffs_file,
    source_order_items_file
)
select
  p.customer_id,
  c.first_name,
  c.last_name,
  c.phone,
  c.email,
  p.order_date,
  p.quantity,
  p.store_name,
  c.source_customers_file,
  c.source_customers_file,
  source_orders_file,
  source_store_file,
  source_staffs_file,
  source_order_items_file
from
  tmp_silver_customers c
    JOIN pending p
      ON c.customer_id = p.customer_id

In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit, year, month

current_ts = current_timestamp()

df_orders_pending_gold = spark.sql("""

with pending as (
  select
    customer_id,
    order_date,
    sum(quantity) as quantity,
    store_name,
    source_orders_file,
    source_store_file,
    source_staffs_file,
    source_order_items_file
  -- p.order_status
  from
    tmp_silver_orders
  where
    1 = 1
    AND upper(order_status) = "PENDING"
  group by
    customer_id,
    order_date,
    store_name,
    source_orders_file,
    source_store_file,
    source_staffs_file,
    source_order_items_file
)
select
  p.customer_id,
  c.first_name,
  c.last_name,
  c.phone,
  c.email,
  p.order_date,
  p.quantity,
  p.store_name,
  c.source_customers_file,
  source_orders_file,
  source_store_file,
  source_staffs_file,
  source_order_items_file
from
  tmp_silver_customers c
    JOIN pending p
      ON c.customer_id = p.customer_id   
""")\
  .withColumn("ingestion_timestamp", current_ts) \
  .withColumn("created_at", current_ts) \
  .withColumn("updated_at", lit(None).cast("timestamp"))

In [0]:
%python
display(df_orders_pending_gold)

In [0]:
%python
#salvar em parquet como delta na camada silver
df_orders_pending_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{gold_path}.orders_pending")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bikestore.logistics.orders_pending;

In [0]:
%sql
CREATE OR REPLACE TABLE bikestore.logistics.orders_pending AS
SELECT * FROM bikestore.gold.orders_pending;

In [0]:
%sql
select * from bikestore.logistics.orders_pending;